# Monthly Churn Analysis
**Purpose:** Monthly churn rate trends, month-of-year cross-year patterns, and churn reasons broken down by month.

In [0]:
%run ./NB01_Config_and_Setup

## 1 · Rebuild model_df (or load from Delta)

In [0]:
section("Loading and labelling billings")
billings = load_table(BILLINGS_TABLE, BILLINGS_PATH)
billings = parse_dates(billings, ['Prospect_Renewal_Date','Closed_Date'])
billings['days_to_close'] = (billings['Closed_Date'] - billings['Prospect_Renewal_Date']).dt.days
conditions = [
    (billings['Prospect_Outcome']=='Churned') & (billings['Closed_Date'].isna()),
    (billings['Prospect_Outcome']=='Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    (billings['Prospect_Outcome']=='Churned') & (billings['days_to_close'] < 0),
    billings['Prospect_Outcome']=='Won',
    billings['Prospect_Outcome']=='Open',
]
billings['Churn_Label'] = np.select(conditions,
    ['Churned','Churned','Pre_Churned','Won','Open'], default='Churned')
model_df = billings[billings['Churn_Label'].isin(['Won','Churned'])].copy()
model_df['Target']        = (model_df['Churn_Label']=='Churned').astype(int)
model_df['Renewal_Year']  = model_df['Prospect_Renewal_Date'].dt.year
model_df['Renewal_Month'] = model_df['Prospect_Renewal_Date'].dt.to_period('M').astype(str)
model_df['Month_Num']     = model_df['Prospect_Renewal_Date'].dt.month
model_df['Month_Name']    = model_df['Prospect_Renewal_Date'].dt.strftime('%b')
mdf = model_df[model_df['Renewal_Year'].isin(TARGET_YEARS)].copy()
print(f"[NB04] Analysis population: {len(mdf):,} records")


## 2 · Monthly Churn Rate Trend

In [0]:
section("Monthly Churn Rate — All Years")
monthly = mdf.groupby('Renewal_Month').agg(
    Total=('Target','count'), Churned=('Target','sum')
).reset_index().sort_values('Renewal_Month')
monthly['Churn_Rate'] = (monthly['Churned']/monthly['Total']*100).round(2)
monthly['Year'] = monthly['Renewal_Month'].str[:4].astype(int)
display_df(monthly, "Monthly Churn Summary")


In [0]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=False)
fig.suptitle("Monthly Churn Rate by Year", fontsize=15, y=1.01)

for i, yr in enumerate(TARGET_YEARS):
    yr_data = monthly[monthly['Year']==yr].copy()
    bars = axes[i].bar(yr_data['Renewal_Month'].str[5:],
                       yr_data['Churn_Rate'],
                       color=[rate_color(r) for r in yr_data['Churn_Rate']],
                       edgecolor='#0a0e1a', width=0.6)
    axes[i].axhline(yr_data['Churn_Rate'].mean(), color='white', linestyle='--',
                    linewidth=1, alpha=0.5, label=f"Avg {yr_data['Churn_Rate'].mean():.1f}%")
    axes[i].set_title(f"{yr} — Peak: {yr_data['Churn_Rate'].max():.2f}% ({yr_data.loc[yr_data['Churn_Rate'].idxmax(),'Renewal_Month']})",
                      color=year_color(yr))
    axes[i].set_ylabel("Churn Rate (%)")
    axes[i].legend()
    # Annotate bars
    for bar, row in zip(bars, yr_data.itertuples()):
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                     f"{row.Churn_Rate:.1f}%", ha='center', va='bottom', fontsize=7.5)
plt.tight_layout(); plt.show()


## 3 · Month-of-Year Analysis (All Years Overlaid)

In [0]:
section("Month-of-Year Cross-Year Comparison")
month_yr = mdf.groupby(['Month_Num','Renewal_Year']).agg(
    Total=('Target','count'), Churned=('Target','sum')
).reset_index()
month_yr['Churn_Rate'] = (month_yr['Churned']/month_yr['Total']*100).round(2)
month_yr['Month_Name'] = pd.to_datetime(month_yr['Month_Num'], format='%m').dt.strftime('%b')

# Pivot for display
pivot_rate = month_yr.pivot(index='Month_Name', columns='Renewal_Year', values='Churn_Rate')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot_rate = pivot_rate.reindex(month_order)
display_df(pivot_rate.reset_index(), "Churn Rate % by Month (row) and Year (col)")


In [0]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_title("Month-of-Year Churn Rate — 2023 vs 2024 vs 2025", fontsize=13)
x = np.arange(12)
width = 0.26
for i, yr in enumerate(TARGET_YEARS):
    vals = [month_yr[(month_yr['Month_Num']==m+1)&(month_yr['Renewal_Year']==yr)]['Churn_Rate'].values
            for m in range(12)]
    vals = [v[0] if len(v)>0 else 0 for v in vals]
    ax.bar(x + i*width, vals, width, label=str(yr), color=year_color(yr), edgecolor='#0a0e1a', alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(month_order)
ax.set_ylabel("Churn Rate (%)")
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


In [0]:
# Heatmap: month vs year
fig, ax = plt.subplots(figsize=(10, 5))
heat_data = month_yr.pivot(index='Renewal_Year', columns='Month_Num', values='Churn_Rate')
heat_data.columns = month_order
im = ax.imshow(heat_data.values, cmap='RdYlGn_r', aspect='auto',
               vmin=heat_data.values.min(), vmax=heat_data.values.max())
ax.set_xticks(range(12)); ax.set_xticklabels(month_order)
ax.set_yticks(range(len(TARGET_YEARS))); ax.set_yticklabels(TARGET_YEARS)
for r in range(len(TARGET_YEARS)):
    for c in range(12):
        val = heat_data.values[r,c]
        ax.text(c, r, f"{val:.1f}%", ha='center', va='center', fontsize=9,
                color='white' if val>10 else '#111827')
plt.colorbar(im, ax=ax, label='Churn Rate %')
ax.set_title("Churn Rate Heatmap — Month × Year", fontsize=13)
plt.tight_layout(); plt.show()


## 4 · Churn Reasons by Month

In [0]:
section("Churn Reasons by Month")
churn_only = model_df[model_df['Target']==1].copy()
churn_only['Renewal_Month'] = churn_only['Prospect_Renewal_Date'].dt.to_period('M').astype(str)

# Top churn reasons overall
top_reasons = churn_only['Prospect_Status'].value_counts().head(8).index.tolist()

monthly_reasons = churn_only[churn_only['Prospect_Status'].isin(top_reasons)]    .groupby(['Renewal_Month','Prospect_Status']).size().reset_index(name='Count')
monthly_reasons = monthly_reasons[monthly_reasons['Renewal_Month'].str[:4].isin([str(y) for y in TARGET_YEARS])]

display_df(monthly_reasons.pivot_table(
    index='Renewal_Month', columns='Prospect_Status', values='Count', fill_value=0).reset_index(),
    "Monthly Churn Counts by Reason")


In [0]:
# Stacked bar: monthly reasons
reason_pivot = monthly_reasons.pivot_table(
    index='Renewal_Month', columns='Prospect_Status', values='Count', fill_value=0)
reason_pivot = reason_pivot.sort_index()

fig, ax = plt.subplots(figsize=(18, 7))
reason_colors = [PALETTE['danger'],'#dc2626','#b91c1c',PALETTE['amber'],'#d97706',
                 PALETTE['blue'],'#2563eb',PALETTE['green']]
reason_pivot.plot(kind='bar', stacked=True, ax=ax,
                  color=reason_colors[:len(reason_pivot.columns)],
                  edgecolor='#0a0e1a', width=0.75)
ax.set_title("Monthly Churn Volume by Reason", fontsize=13)
ax.set_xlabel("Renewal Month")
ax.set_ylabel("Churned Accounts")
ax.tick_params(axis='x', rotation=45)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()


In [0]:
# Top reason per month
top_reason_by_month = churn_only[churn_only['Renewal_Month'].str[:4].isin([str(y) for y in TARGET_YEARS])]    .groupby(['Renewal_Month','Prospect_Status'])    .size().reset_index(name='Count')    .sort_values(['Renewal_Month','Count'], ascending=[True,False])    .drop_duplicates('Renewal_Month')
top_reason_by_month = top_reason_by_month[['Renewal_Month','Prospect_Status','Count']]
top_reason_by_month.columns = ['Month','Top Churn Reason','Count']
display_df(top_reason_by_month, "Dominant Churn Reason per Month")


## 5 · Statistical Seasonality Test

In [0]:
section("Kruskal-Wallis Test — Is month-of-year a significant predictor?")
monthly_rates_by_month = month_yr[month_yr['Renewal_Year'].isin(TARGET_YEARS)]
groups = [monthly_rates_by_month[monthly_rates_by_month['Month_Num']==m]['Churn_Rate'].values
          for m in range(1,13)]
groups = [g for g in groups if len(g)>0]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H-statistic : {stat:.3f}")
print(f"p-value                    : {p:.4f}")
if p < 0.05:
    print("  RESULT: Significant seasonal pattern — month IS a predictor of churn rate.")
else:
    print("  RESULT: No significant seasonal effect detected.")
print()
print("Months ranked by mean churn rate:")
print(monthly_rates_by_month.groupby('Month_Num')['Churn_Rate'].mean().sort_values(ascending=False).to_string())
